# Second-stage classification of first-stage beach surfaces

This notebook reads first-stage globally weighted ensemble classifications, selects cells predicted as class 1 or 2, and assigns second-stage grain-size classes 11, 12, or 13. It loads the four best second-stage models and combines their probabilities using global per-model weights derived from inner out-of-fold predictions during all-data deployment fitting.

Candidate cell IDs come directly from `Results/bs_all_cells_classification/weighted_ensemble/<date>/classification.csv`, so the raw data directory is not scanned. Matrix data and same-date context neighbors are loaded on demand from `Data/Raw/beach_all/<date>/`. Results are written to `Results/gs_all_cells_classification/weighted_ensemble/<date>/classification.csv`.

For the nested-CV workflow, this notebook loads the final deployment checkpoints (trained on all labelled grain-size cells), not an outer-fold checkpoint. Re-evaluate the grain-size ensembles after retraining a deployment model so its saved weights and model metadata agree.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.ensemble import (
    classify_date_directory_with_global_ensemble,
    load_global_weighted_ensemble,
    load_model_artifact,
    validate_artifact_compatibility,
)
from Source.training import get_device

In [ ]:
SOURCE_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
FIRST_STAGE_CLASSIFICATION_ROOT = (
    PROJECT_ROOT / "Results" / "bs_all_cells_classification" / "weighted_ensemble"
)
SECOND_STAGE_ENSEMBLE_DIR = PROJECT_ROOT / "Results" / "gs_ensemble"
ENSEMBLE_DEFINITION_PATH = SECOND_STAGE_ENSEMBLE_DIR / "ensemble_definition.json"
OUTPUT_ROOT = (
    PROJECT_ROOT / "Results" / "gs_all_cells_classification" / "weighted_ensemble"
)

MODEL_RESULT_DIRS = {
    "convnext_tiny": PROJECT_ROOT / "Results" / "gs_convnext_tiny",
    "vit_tiny": PROJECT_ROOT / "Results" / "gs_vit_tiny",
    "swin_tiny": PROJECT_ROOT / "Results" / "gs_swin_tiny",
    "efficientnetv2_s": PROJECT_ROOT / "Results" / "gs_efficientnetv2_s",
}

FIRST_STAGE_SURFACE_CLASSES = {"1", "2"}
EXPECTED_SECOND_STAGE_CLASSES = {"11", "12", "13"}
BATCH_SIZE = 64
NUM_WORKERS = 0
BASE_SUFFIX = "r"

print(f"Project root:                {PROJECT_ROOT}", flush=True)
print(f"Raw cell data:               {SOURCE_ROOT}", flush=True)
print(f"First-stage classifications: {FIRST_STAGE_CLASSIFICATION_ROOT}", flush=True)
print(f"Second-stage ensemble:       {ENSEMBLE_DEFINITION_PATH}", flush=True)
print(f"Output root:                 {OUTPUT_ROOT}", flush=True)

In [ ]:
print("Checking inputs and second-stage model artifacts...", flush=True)
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"Cell data directory not found: {SOURCE_ROOT}")
if not FIRST_STAGE_CLASSIFICATION_ROOT.exists():
    raise FileNotFoundError(
        f"First-stage classification directory not found: {FIRST_STAGE_CLASSIFICATION_ROOT}"
    )
if not ENSEMBLE_DEFINITION_PATH.exists():
    raise FileNotFoundError(
        f"Second-stage ensemble definition not found: {ENSEMBLE_DEFINITION_PATH}. "
        "Run gs_evaluate_ensemble.ipynb first."
    )

artifacts = [
    load_model_artifact(name, result_dir)
    for name, result_dir in MODEL_RESULT_DIRS.items()
]
validate_artifact_compatibility(artifacts)
reference_metadata = artifacts[0].metadata
idx_to_class_code = {
    int(index): str(code)
    for index, code in reference_metadata["idx_to_class_code"].items()
}
if set(idx_to_class_code.values()) != EXPECTED_SECOND_STAGE_CLASSES:
    raise ValueError(
        "Second-stage model classes do not match the expected grain-size classes: "
        f"{sorted(idx_to_class_code.values())}"
    )

artifact_table = pd.DataFrame([
    {
        "model": artifact.name,
        "context_radius": int(artifact.metadata["context_radius"]),
        "image_size": int(artifact.metadata["image_size"]),
        "checkpoint": str(artifact.checkpoint_path),
    }
    for artifact in artifacts
])
display(artifact_table)

print("Selecting compute device and loading all four second-stage models...", flush=True)
device = get_device()
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(device)}", flush=True)

models, model_weights, ensemble_definition = load_global_weighted_ensemble(
    artifacts,
    ENSEMBLE_DEFINITION_PATH,
    device,
)
print("Loaded second-stage global model weights:", flush=True)
for model_name, weight in model_weights.items():
    print(f"  {model_name}: {weight:.6f}", flush=True)

In [ ]:
print("Locating first-stage per-date classification files...", flush=True)
classification_files = sorted(
    FIRST_STAGE_CLASSIFICATION_ROOT.glob("*/classification.csv"),
    key=lambda path: path.parent.name,
)
if not classification_files:
    raise FileNotFoundError(
        "No first-stage classification files were found under "
        f"{FIRST_STAGE_CLASSIFICATION_ROOT}"
    )

print(f"Date classification files found: {len(classification_files)}", flush=True)
print(
    "Only cell IDs predicted as first-stage class 1 or 2 will be loaded. "
    "The raw date directories will not be scanned.",
    flush=True,
)

In [ ]:
print("Starting second-stage globally weighted ensemble classification...", flush=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_rows = []

for date_index, classification_path in enumerate(classification_files, start=1):
    date = classification_path.parent.name
    print(f"Starting date {date_index}/{len(classification_files)}: {date}", flush=True)
    first_stage_df = pd.read_csv(classification_path, dtype=str, keep_default_na=False)
    required_columns = {"cell_id", "predicted_class"}
    missing_columns = required_columns - set(first_stage_df.columns)
    if missing_columns:
        raise ValueError(
            f"{classification_path} is missing columns: {sorted(missing_columns)}"
        )

    if "date" in first_stage_df.columns:
        recorded_dates = set(first_stage_df["date"].astype(str).str.strip()) - {""}
        if recorded_dates != {date}:
            raise ValueError(
                f"Date values in {classification_path} do not match folder {date}: "
                f"{sorted(recorded_dates)}"
            )

    numeric_classes = pd.to_numeric(first_stage_df["predicted_class"], errors="coerce")
    if numeric_classes.isna().any() or ((numeric_classes % 1) != 0).any():
        raise ValueError(f"Non-integer predicted classes found in {classification_path}")
    first_stage_df["_class_code"] = numeric_classes.astype("int64").astype("string")
    selected_df = first_stage_df.loc[
        first_stage_df["_class_code"].isin(FIRST_STAGE_SURFACE_CLASSES)
    ].copy()
    selected_df["cell_id"] = selected_df["cell_id"].astype(str).str.strip()
    if selected_df["cell_id"].eq("").any():
        raise ValueError(f"Empty cell IDs found in {classification_path}")
    selected_df = selected_df.drop_duplicates(subset="cell_id", keep="first")
    candidate_cell_ids = selected_df["cell_id"].tolist()
    class_counts = selected_df["_class_code"].value_counts()

    print(
        f"  Selected {len(candidate_cell_ids):,} cells "
        f"(class 1: {int(class_counts.get('1', 0)):,}, "
        f"class 2: {int(class_counts.get('2', 0)):,})",
        flush=True,
    )

    date_dir = SOURCE_ROOT / date
    if not date_dir.is_dir():
        raise FileNotFoundError(f"Raw date directory not found: {date_dir}")

    if candidate_cell_ids:
        result = classify_date_directory_with_global_ensemble(
            date_dir=date_dir,
            source_root=SOURCE_ROOT,
            output_root=OUTPUT_ROOT,
            artifacts=artifacts,
            models=models,
            weights=model_weights,
            idx_to_class_code=idx_to_class_code,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            base_suffix=BASE_SUFFIX,
            device=device,
            candidate_cell_ids=candidate_cell_ids,
        )
    else:
        output_dir = OUTPUT_ROOT / date
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / "classification.csv"
        pd.DataFrame(columns=["date", "cell_id", "predicted_class"]).to_csv(
            output_path,
            index=False,
        )
        result = {
            "date": date,
            "classified_cells": 0,
            "incomplete_candidates": 0,
            "device": str(device),
            "output_path": output_path,
        }

    result["first_stage_rows"] = len(first_stage_df)
    result["selected_candidates"] = len(candidate_cell_ids)
    result["first_stage_class_1"] = int(class_counts.get("1", 0))
    result["first_stage_class_2"] = int(class_counts.get("2", 0))
    summary_rows.append(result)
    print(
        f"Finished {date}: classified {result['classified_cells']:,}, "
        f"missing/incomplete {result['incomplete_candidates']:,}, "
        f"saved {result['output_path']}",
        flush=True,
    )

classification_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_ROOT / "classification_summary.csv"
classification_summary.to_csv(summary_path, index=False)
print("Second-stage classification finished.", flush=True)
print(f"Saved summary: {summary_path}", flush=True)
classification_summary